<a href="https://colab.research.google.com/github/blancavazquez/PLN/blob/2027-01/notebooks/Metrics_BLEU%2C_ROUGE_y_BERTScore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BLEU, ROUGE y BERTScore

Son métricas que se utilizan en el Procesamiento del Lenguaje Natural (PLN) y en Inteligencia Artificial para evaluar qué tan bien un texto generado por una máquina se parece a un texto de referencia escrito por un humano.



## ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

* Qué mide: La exhaustividad (recall), es decir, qué porcentaje de la información o de las palabras del texto de referencia logró capturar el texto generado.
* Para qué sirve: Es la métrica estándar para tareas de resumen de textos (text summarization).
* Limitación: Al igual que BLEU, depende de coincidencias superficiales de palabras y puede ignorar el sentido si cambia la redacción

![Alt text](https://miro.medium.com/v2/resize:fit:2000/format:webp/1*gpxcM-ylaUfp00wR1931WQ.png)

ROUGE-N (Superposición de N-gramas)

$$
Recall = \frac{Ngramas\_coincidentes}{total\_Ngramas\_referencia}\
$$

$$
Precision = \frac{Ngramas\_coincidentes}{total\_Ngramas\_candidato}\
$$

$$
F1-score = \frac{2 X Precision X Recall}{Precision+Recall}
$$



------------------------

In [ ]:
!pip install nltk evaluate rouge-score bert-score torch --quiet

In [ ]:
import evaluate
from rouge_score import rouge_scorer

from bert_score import BERTScorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Textos de ejemplo
reference = "The cat is sitting on the mat"
candidate = "The cat is on the mat"

print("*******************************")
print("Reference: ", reference)
print("Candidate: ", candidate)
print("*******************************")

print("\n*ROUGE Implementation ")
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'],use_stemmer=True)
rouge_scores = scorer.score(reference, candidate)

for key, value in rouge_scores.items():
    print(f"{key}")
    print(f"Precision: {value.precision:.4f}")
    print(f"Recall: {value.recall:.4f}")
    print(f"F1 Score: {value.fmeasure:.4f}")
    print()

print("\n*ROUGE (Hugging Face Evaluate)")
rouge_score = evaluate.load("rouge")
scores = rouge_score.compute(predictions=[candidate], references=[reference])
print(scores)

## BLEU

BLEU (Bilingual Evaluation Understudy): Mide la precisión de n-gramas (coincidencia exacta de palabras y orden). Ideal para traducción automática. Es rápido pero ciego a los sinónimos.


Step-by-Step Example

* Reference (Target): The NASA Opportunity rover is battling a massive dust storm on Mars. (11 tokens)

* Candidate (Generated): A NASA rover is fighting a massive storm on Mars. (11 tokens)

**1. N-Gram Precision (p_{n})**

Unigram to 4-gram precision counts are calculated with clipping (capping the count of each n-gram to its maximum occurrence in the reference):
* Precision 1 (1-gram): 9 / 11 ≈ 0.818
* Precision 2 (2-gram): 5 / 10 = 0.500
* Precision 3 (3-gram): 2 / 9 ≈ 0.222
* Precision 4 (4-gram): 1 / 8 = 0.125


**2. Geometric Mean of Precisions**

We take the geometric mean of the four precision scores (using equal weights \(w_n = 1/4\)):

$$
Geometric\ Mean=(p_{1}\times p_{2}\times p_{3}\times p_{4})^{\frac{1}{4}}
$$

$$
Geometric\ Mean=(0.818\times 0.500\times 0.222\times 0.125)^{\frac{1}{4}}=(0.01136)^{\frac{1}{4}}\approx 0.326
$$


**3. Brevity Penalty (BP)**

The brevity penalty prevents short translations from getting artificially high precision scores.
* Candidate length (c) = 11
* Reference length (r) = 11
* Since c ≥ r, \(BP = \exp(1 - 11/11) = \exp(0) = 1.0\)


**4. Final BLEU Score**

$$
\text{BLEU}=BP\times \text{Geometric Mean}
$$

$$
\text{BLEU}=1.0\times 0.326=0.33 {\ (or\ }33\%\text{)}
$$


**Puntaje BLEU	- Interpretación práctica**
* Menos de 20%	Prácticamente inútil. Palabras sueltas al azar.
* 20% - 40%	Comprensible, pero con errores gramaticales graves y mal flujo.
* 40% - 60%	Buena traducción.
* Más de 60%	Excelente calidad. A menudo mejor o igual que la traducción de un humano promedio.


![Alt text](https://miro.medium.com/v2/resize:fit:2000/format:webp/1*acEFlTN8MZKyz9G9LWljIw.png)

![Alt text](https://miro.medium.com/v2/resize:fit:2000/format:webp/1*9CB6NZfwXXWcgb2EyQPK1w.png)

In [ ]:
import evaluate
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Textos de ejemplo
reference = "The cat is sitting on the mat"
candidate = "The cat is on the mat"

# --- 2. BLEU (NLTK) ---
ref_tokens = [reference.lower().split()]
cand_tokens = candidate.lower().split()
smoothie = SmoothingFunction().method1
bleu_score = sentence_bleu(ref_tokens, cand_tokens, smoothing_function=smoothie)

print(f"BLEU Score:      {bleu_score:.4f}")

------------------

## BERTscore

Utiliza representaciones contextuales de modelos tipo BERT para calcular la similitud semántica mediante coseno. Entiende sinónimos y paráfrasis (por ejemplo, "dog" vs "hound" o "car" vs "automobile"). Es más lento porque requiere inferencia de un modelo de lenguaje.

![Alt text](https://miro.medium.com/v2/resize:fit:2000/format:webp/1*SRY_emqy8H7vdbOx3KYANw.png)

In [ ]:
from bert_score import BERTScorer

# Textos de ejemplo
reference = "The cat is sitting on the mat"
candidate = "The cat is on the mat"

scorer = BERTScorer(model_type='bert-base-uncased',
                    lang="en",
                    rescale_with_baseline=True)
P, R, F1 = scorer.score([candidate], [reference])

print("**********************")
print(f"BERTScore Precision:    {P.mean().item():.4f}")
print(f"BERTScore Recall:    {R.mean().item():.4f}")
print(f"BERTScore F1:    {F1.mean().item():.4f}")

![Alt text](https://miro.medium.com/v2/resize:fit:2000/format:webp/1*PEg1odwCNvIj0yJedqsWzA.png)

In [ ]:
import evaluate
from bert_score import BERTScorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Textos de ejemplo
reference = "The cat is sitting on the mat"
candidate = "The cat is on the mat"

# --- 1. ROUGE (Hugging Face Evaluate) ---
rouge_metric = evaluate.load("rouge")
rouge_res = rouge_metric.compute(predictions=[candidate], references=[reference])

# --- 2. BLEU (NLTK) ---
ref_tokens = [reference.lower().split()]
cand_tokens = candidate.lower().split()
smoothie = SmoothingFunction().method1
bleu_score = sentence_bleu(ref_tokens, cand_tokens, smoothing_function=smoothie)

# --- 3. BERTScore ---
scorer = BERTScorer(model_type='bert-base-uncased',
                    lang="en",
                    rescale_with_baseline=True)
P, R, F1 = scorer.score([candidate], [reference])

print("**********************")
print(f"ROUGE-1 F1:      {rouge_res['rouge1']:.4f}")
print(f"ROUGE-L F1:      {rouge_res['rougeL']:.4f}")
print(f"BLEU Score:      {bleu_score:.4f}")
print(f"BERTScore Precision:    {P.mean().item():.4f}")
print(f"BERTScore Recall:    {R.mean().item():.4f}")
print(f"BERTScore F1:    {F1.mean().item():.4f}")

Las imágenes presentes en esta libreta fueron tomadas de:

Lahirukavikara. (2026, Agosto 18). ROUGE, BLEU, and BERTScore: How Do We Actually Measure Whether a Foundation Model Gave a Good Answer?. Medium. https://medium.com/beyond-the-algorithm/rouge-bleu-and-bertscore-how-do-we-actually-measure-whether-a-foundation-model-gave-a-good-e4e2e89be700
